# **Processamento de Linguagem Natural [2025-Q3]**
Prof. Alexandre Donizeti Alves

### **PROJETO PRÁTICO** [LangChain + Grandes Modelos de Linguagem]


### **EQUIPE**

---




**Integrante 01:**

`Erik Farias Lima - 11202111817`

**Integrante 02:**

`Felipe Moya de Carvalho Olivalves - 11202111294`


### **GRANDE MODELO DE LINGUAGEM (*Large Language Model - LLM*)**

---



>


**LLM**: `Gemini`

>

**Link para a documentação oficial**: https://ai.google.dev/gemini-api/docs?hl=pt-br



### **API (Opcional)**
---


**Não utilizamos**









### **DESCRIÇÃO**
---

Implementação de um `notebook` no `Google Colab` que utiliza do framework **`LangChain`** e do **LLM** Gemini da Google, em sua versão 2.5 Flash, e aplica duas técnicas de PLN: Tradução de textos e Análise de sentimentos, em reviews de jogos extraidas do site `Metacritic`, página na Web destinada a publicação de críticas/resenhas de jogos, filmes e outras midías.

>
Técnicas de PLN que foram aplicadas nesse projeto:

*   Análise de Sentimentos;
*   Tradução de Textos.
>



### **IMPLEMENTAÇÃO**
---

In [ ]:
# Instalação da biblioteca de integração com o LangChain
!pip install -qU langchain-google-genai
# Instalação da biblioteca BeautifulSoup para scraping em paginas da Web
!pip install beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 14.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.9.0 which is incompatible.


In [ ]:
# Definindo a chave da API
from getpass import getpass

GOOGLE_API_KEY = getpass()

··········


In [ ]:
# Chamando a interface de interação com o chat do Gemini
from langchain_google_genai import ChatGoogleGenerativeAI
# Template de prompt para Chat Models
from langchain_core.prompts import ChatPromptTemplate

# Definindo o modelo
modelo = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key = GOOGLE_API_KEY)

# Definindo o comportamento da IA
system_template = "Traduza o texto, não apresente opções distintas, escolha uma sozinho"

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", system_template),
        ("user", "{texto}")
    ]
)

# Encadeamento de componentes para conectar o prompt ao modelo LLM
chain = prompt_template | modelo


In [ ]:
# Definindo o comportamento da IA
system_template2 = "Realize uma analise dos sentimentos expressados pelo analista, seja breve, explique em um senteça curta, comece dizendo se é uma analise positiva, negativa ou neutra"

prompt_template2 = ChatPromptTemplate.from_messages(
    [
        ("system", system_template2),
        ("user", "{texto}")
    ]
)

# Encadeamento de componentes para conectar o prompt ao modelo LLM
chain2 = prompt_template2 | modelo


In [ ]:
# Biblioteca para tratar requisições http
import requests
# Biblioteca para realizar scraping em paginas da Web
from bs4 import BeautifulSoup

def reviewAnalysis(url):
  # Informa agente de usuario do navegador para que o site permita o acesso
  headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36 Edg/142.0.0.0'}
  response = requests.get(f"{url}critic-reviews/", headers=headers)

  # Checa se a requisicao http foi bem-sucedida
  if response.status_code == 200:
    htmlContent = response.text
    soup = BeautifulSoup(htmlContent, "html.parser")
  else:
    print(f"Failed to retrieve the webpage. Status code: {response.status_code}")

  reviews = soup.select("div.c-siteReview_main.g-inner-spacing-medium")

  # Percorre as primeiras 5 reviews
  for r in reviews[:5]:
      # Nome do crítico ou publicação
      critic = r.select_one(".c-siteReviewHeader_publicationName")
      critic_name = critic.text.strip() if critic else "N/A"

      # Nota da review
      score = r.select_one(".c-siteReviewScore_medium, .c-siteReviewScore")
      score_value = score.text.strip() if score else "N/A"

      # Texto da review
      review_text = r.select_one(".c-siteReview_quote, .c-siteReview_body")
      review_content = review_text.text.strip() if review_text else "N/A"
      textoPrompt = review_content

      resultado = chain.invoke({"texto": textoPrompt})
      resultado2 = chain2.invoke({"texto": textoPrompt})


      print(f"Crítico/Publicação: {critic_name}")
      print(f"Nota: {score_value}")
      print(f"Review original: {review_content}")
      print(f"Tradução: {resultado.content}")
      print(f"Analise de sentimentos: {resultado2.content}")
      print("-" * 100)

In [ ]:
reviewAnalysis('https://www.metacritic.com/game/hollow-knight-silksong/')

Crítico/Publicação: GamingBolt
Nota: 100
Review original: Challenging, frustrating, invigorating, and oh-so fulfilling, Silksong is simply a masterpiece in almost every way.
Tradução: Desafiador, frustrante, revigorante e tão gratificante, Silksong é simplesmente uma obra-prima em quase todos os sentidos.
Analise de sentimentos: Positiva, pois, apesar dos desafios e frustrações, o analista sente-se revigorado, realizado e considera o jogo uma obra-prima.
----------------------------------------------------------------------------------------------------
Crítico/Publicação: Gamereactor UK
Nota: 100
Review original: In my nearly ten years here at Gamereactor, I have so far managed to award three perfect scores, and now it's time again. Hollow Knight: Silksong is a masterpiece, simply put. No question about it. Expectations were sky-high in advance, but with a lot of patience and even more skill, Team Cherry has managed to rise above the competition and cement its place in the starry sky 

In [ ]:
reviewAnalysis('https://www.metacritic.com/game/the-lord-of-the-rings-gollum/')

Crítico/Publicação: IGN Adria
Nota: 70
Review original: Surprisingly competent 3D stealth platformer that has managed to accurately showcase one of the most famous anti-heroes in the world of fantasy entertainment. Despite its underdeveloped visual presentation, it is still one of the rare Lord of the Rings games that are worth playing.
Tradução: Um jogo de plataforma 3D de furtividade surpreendentemente competente que conseguiu retratar fielmente um dos mais famosos anti-heróis no mundo do entretenimento de fantasia. Apesar de sua apresentação visual subdesenvolvida, ainda é um dos poucos jogos do Senhor dos Anéis que valem a pena jogar.
Analise de sentimentos: A análise é positiva, pois elogia a competência e o valor do jogo, considerando-o um dos poucos de Senhor dos Anéis que valem a pena, apesar de suas falhas visuais.
----------------------------------------------------------------------------------------------------
Crítico/Publicação: GamingTrend
Nota: 70
Review original: Just 

In [ ]:
reviewAnalysis('https://www.metacritic.com/game/persona-5-the-phantom-x/')

Crítico/Publicação: But Why Tho?
Nota: 90
Review original: The combat flows quickly, the story grabs you and doesn’t let go, and the social systems remain engaging and fun. While the gacha systems will be enough to turn people off, so far, I’m having a blast with Persona5: The Phantom X and will keep coming back for more.
Tradução: O combate flui rapidamente, a história te prende e não te larga mais, e os sistemas sociais continuam envolventes e divertidos. Embora os sistemas de gacha sejam suficientes para afastar algumas pessoas, até agora, estou me divertindo muito com Persona 5: The Phantom X e continuarei voltando para mais.
Analise de sentimentos: É uma análise **positiva**, pois o analista expressa grande entusiasmo e diversão com o jogo, apesar de reconhecer o lado negativo dos sistemas de gacha para outros.
----------------------------------------------------------------------------------------------------
Crítico/Publicação: DBLTAP
Nota: 80
Review original: Persona 5: The Pha